> Notebook-friendly copy of `part-II/4.2-dimensionality-reduction-exercises.ipynb`, generated by `tools/make_live.py`. Edit the book notebook, not this file.

# 4.2) (Exercise) Dimensionality Reduction

<img src='https://github.com/gse-unil/2026_MLEES_book/blob/main/live/part-II/_static/datacenter.webp?raw=1'>
<center> Caption: <i>Denise diagnoses an overheated CPU at our data center in The Dalles, Oregon. <br> For more than a decade, we have built some of the world's most efficient servers.</i> <br> Photo from the <a href='https://www.google.com/about/datacenters/gallery/'>Google Data Center gallery</a> </center>

*Our world is increasingly filled with data from all sorts of sources, including environmental data. Can we reduce the data to a reduced, meaningful space to save on computation time and increase explainability?*

**🎯 Learning objectives**

- Reduce data to a smaller representation that keeps its structure, and see what that gains
- Train a random forest and a logistic regression and compare their training times
- Apply principal component analysis (PCA) from scikit-learn

This notebook will be used in the lab session for week 4 of the course, covers Chapter 8 of the old edition of Géron's book (Chapter 7, "Dimensionality Reduction", in the current *Hands-On Machine Learning with Scikit-Learn and PyTorch* edition), and builds on the [notebooks made available on _Github_](https://github.com/ageron/handson-ml2).

Need a reminder of last week's labs? Click [_here_](https://colab.research.google.com/github/tbeucler/2022_ML_Earth_Env_Sci/blob/main/Lab_Notebooks/Week_3_Decision_Trees_Random_Forests_SVMs.ipynb) to go to notebook for week 3 of the course.

**Notebook Setup**

First, let's import a few common modules, ensure MatplotLib plots figures inline and prepare a function to save the figures. We also check that Python 3.5 or later is installed (although Python 2.x may work, it is deprecated so we strongly recommend you use Python 3 instead), as well as Scikit-Learn ≥0.20.

In [29]:
# Common imports
import numpy as np
import os

# to make this notebook's output stable across runs
rnd_seed = 42
rnd_gen = np.random.default_rng(rnd_seed)

# To plot pretty figures
%matplotlib inline
import matplotlib as mpl
import matplotlib.pyplot as plt
mpl.rc('axes', labelsize=14)
mpl.rc('xtick', labelsize=12)
mpl.rc('ytick', labelsize=12)

# Where to save the figures -- shared _files/ dir for this part's generated output
IMAGES_PATH = "_files"
os.makedirs(IMAGES_PATH, exist_ok=True)

def save_fig(fig_id, tight_layout=True, fig_extension="png", resolution=300):
    path = os.path.join(IMAGES_PATH, fig_id + "." + fig_extension)
    print("Saving figure", fig_id)
    if tight_layout:
        plt.tight_layout()
    plt.savefig(path, format=fig_extension, dpi=resolution)

Dimensionality Reduction using PCA
- permet de réduire le temps et la mémoire 784 variable (nombre de pixel qui décrivent chaque image = colonne) à 150 qui sont des combinaison de de pixel (composante principale) qui résument la forme du chiffre. 
- limite le buit et le surapprentissage, les axes qui expliquent très peu de variance contienne surtout du bruit, c'est la malédication
de trop de dimension : plus il y a des variable plus il faut de donnée pour bien généraliser 


This week we'll be looking at how to reduce the dimensionality of a large dataset in order to improve our classifying algorithm's performance! With that in mind, let's being the exercise by loading the MNIST dataset.

## Q1) Load the input features and truth variable into X and y, then split the data into a training and test dataset using scikit's train_test_split method. Use *test_size=0.15*, and remember to set the random state to *rnd_seed!*

*Hint 1: The `'data'` and `'target'` keys for mnist will return X and y.*

*Hint 2: [Here's the documentation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) for train/test split.*

In [30]:
# Chargement du jeu de données MNIST (70 000 images de chiffres manuscrits 28x28 pixels)
# fetch_openml télécharge le dataset depuis OpenML (internet nécessaire la 1ère fois).
# as_frame=False -> on récupère des tableaux NumPy (et pas un DataFrame pandas)
from sklearn.datasets import fetch_openml
mnist = fetch_openml('mnist_784', version=1, as_frame=False)

In [31]:
# Lecture des images et des chiffres correspondants
# ---------------------------------------------------------------------------
# mnist est un objet "Bunch" (une sorte de dictionnaire) avec les clés 'data' et 'target'
X = mnist["data"]    # shape (70000, 784) : chaque image 28x28 est "aplatie" en 784 pixels (valeurs de 0 à 255)
y = mnist["target"]  # shape (70000,)     : le chiffre à prédire (0 à 9), stocké sous forme de texte (ex: '5')

print(X.shape, y.shape)  # petite vérification des dimensions

(70000, 784) (70000,)


💡 **Explication** : MNIST contient des images 28×28 pixels, soit 784 valeurs par image. Chaque pixel est donc une *variable* (feature). C'est cette dimension de 784 que la PCA va chercher à réduire.

In [32]:
# Import de train_test_split() pour créer les jeux d'entraînement et de test
# ---------------------------------------------------------------------------
from sklearn.model_selection import train_test_split

# Séparation de X et y en jeux d'entraînement et de test
# ---------------------------------------------------------------------------
# test_size=0.15    -> 15 % des données (10 500 images) vont dans le jeu de test, 85 % (59 500) à l'entraînement
# random_state=rnd_seed -> graine aléatoire fixée pour que le découpage soit reproductible
# Attention à l'ordre des sorties : X_train, X_test, y_train, y_test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=rnd_seed)

We now once again have a training and testing dataset with which to work with. Let's try training a random forest tree classifier on it. You've had experience with them before, so let's have you import the `RandomForestClassifier` from sklearn and instantiate it.

## Q2) Import the `RandomForestClassifier` model from sklearn. Then, instantiate it with 100 estimators and set the random state to *rnd_seed!*

*Hint 1: [Here's the documentation](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html) for `RandomForestClassifier`*

*Hint 2: [Here's the documentation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) for train/test split.*

*Hint 3: If you're still confused about **instantiation**, there's a [blurb on wikipedia](https://en.wikipedia.org/wiki/Instance_(computer_science)) describing it in the context of computer science.*

In [33]:
# Import de RandomForestClassifier (il se trouve dans le module sklearn.ensemble)
# ---------------------------------------------------------------------------
from sklearn.ensemble import RandomForestClassifier

In [34]:
# Ici on crée (= on "instancie") un classifieur Random Forest avec nos réglages : 100 arbres, random_state=rnd_seed
# Rien n'est entraîné à ce stade : on crée juste l'objet avec ses hyperparamètres.
# ------------------------------------------------------------------------------------------------------
rnd_clf = RandomForestClassifier(n_estimators=100,      # Nombre d'arbres (estimators)
                                 random_state=rnd_seed) # Graine aléatoire (résultats reproductibles)

We're now going to measure how quickly the algorithm is fitted to the mnist dataset! To do this, we'll have to import the `time` library. With it, we'll be able to get a timestamp immediately before and after we fit the algorithm, and we'll get the time by calculating the difference.

## Q3) Import the time library and calculate how long it takes to fit the `RandomForestClassifier` model.

*Hint 1: [Here's the documentation](https://docs.python.org/3/library/time.html#time.time) to the function used for getting timestamps*

*Hint 2: [Here's the documentation](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html#sklearn.ensemble.RandomForestClassifier.fit) for the fitting method used in `RandomForestClassifier`.*

In [35]:
import time

In [36]:
# time.time() renvoie l'heure actuelle en secondes : on la relève AVANT l'entraînement, pour voir si éla réduction fait vraiment gagner du temps
# ------------------------------------------------------------------------------------------------------
t0 = time.time()

# Entraînement du Random Forest : .fit(entrées, vérité terrain)
# ------------------------------------------------------------------------------------------------------
rnd_clf.fit(X_train, y_train)

# Même fonction pour relever l'heure APRÈS l'entraînement
# ------------------------------------------------------------------------------------------------------
t1 = time.time()

In [37]:
# Run this as is, how many seconds did it take to train the classifier?
# ------------------------------------------------------------------------------------------------------
train_t_rf = t1-t0

print(f"Training took {train_t_rf:.2f}s")
# Ça a pris envrion 33 seconde

Training took 33.06s


💡 **Explication** : `time.time()` donne l'heure en secondes. La différence `t1 - t0` correspond donc à la durée de `.fit()`. Les temps dépendent de ton ordinateur, c'est la *comparaison* entre modèles qui est intéressante.

We care about more than just how long we took to trian the model, however! Let's get an accuracy score for our model.

## Q4) Get an accuracy score for the predictions from the RandomForestClassifier

*Hint 1: [Here is the documentation](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.accuracy_score.html) for the `accuracy_score` metric in sklearn.*

*Hint 2: [Here is the documentation](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html#sklearn.ensemble.RandomForestClassifier.predict) for the predict method in `RandomForestClassifier`*

In [38]:
# Import de la métrique accuracy_score (module sklearn.metrics)
# accuracy = proportion de prédictions correctes = (nb de bonnes prédictions) / (nb total)
# ------------------------------------------------------------------------------------------------------
from sklearn.metrics import accuracy_score

In [39]:
# On utilise le modèle entraîné pour prédire les chiffres du jeu de TEST (jamais vu pendant l'entraînement)
# ------------------------------------------------------------------------------------------------------
y_pred = rnd_clf.predict(X_test)

In [40]:
# accuracy_score(vérité, prédictions) : on compare les vrais chiffres (y_test) aux prédictions (y_pred)
# ------------------------------------------------------------------------------------------------------
rf_accuracy = accuracy_score(y_test, y_pred)

# À lancer tel quel. Nous avions obtenu une accuracy de 96.7 %. Et vous ?
# ------------------------------------------------------------------------------------------------------
print(f"RF Model Accuracy: {rf_accuracy:.2%}") # 96.7%

RF Model Accuracy: 96.70%


Let's try doing the same with with a logistic regression algorithm to see how it compares.

## Q5) Repeat Q2-4 with a logistic regression algorithm using sklearn's `LogisticRegression` class. Hyperparameters: `solver='lbfgs'` (multinomial is the default behavior for multiclass problems with this solver)

*Hint 1: [Here is the documentation](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html) for the `LogisticRegression` class.

In [41]:
# Import de la classe LogisticRegression (module sklearn.linear_model)
# ---------------------------------------------------------------------------
from sklearn.linear_model import LogisticRegression

In [42]:
# Création d'un objet LogisticRegression avec des hyperparamètres choisis
# ---------------------------------------------------------------------------
# solver="lbfgs" : algorithme d'optimisation (gère le cas multiclasse, ici 10 chiffres, en mode "multinomial") = SOFTMAX
# random_state=42 : graine aléatoire pour la reproductibilité
# Remarque : max_iter vaut 100 par défaut. Sur MNIST non normalisé (pixels de 0 à 255), lbfgs
# n'a souvent pas le temps de converger : un "ConvergenceWarning" peut apparaître, c'est normal ici
# (on pourrait l'éviter en normalisant les pixels /255 ou en augmentant max_iter).
log_clf = LogisticRegression(solver="lbfgs",  # Solver
                             random_state=42) # Random State

In [43]:
# Timestamp for **current** time before training the LogisticRegression classifier
# ------------------------------------------------------------------------------------------------------
t0 = time.time()

In [44]:
# Entraînement du classifieur de régression logistique (sur les 784 pixels)
# ------------------------------------------------------------------------------------------------------
log_clf.fit(X_train, y_train)

/Users/estelle/miniforge3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",42
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following a

In [45]:
# Timestamp for **current** time after training the LogisticRegression classifier
# ------------------------------------------------------------------------------------------------------
t1 = time.time()

In [46]:
# Run this as is, how many seconds did it take to train the classifier? 12 seconde
# ------------------------------------------------------------------------------------------------------
train_t_log = t1-t0
print(f"Training took {train_t_log:.2f}s")

Training took 11.83s


In [47]:
# Prédictions sur le jeu de test (X_test) avec la régression logistique
# ------------------------------------------------------------------------------------------------------
y_pred = log_clf.predict(X_test)

# À lancer tel quel. Nous avions obtenu une accuracy de 92.1 %. Et vous ? 92.1 %
# ------------------------------------------------------------------------------------------------------
log_accuracy = accuracy_score(y_test, y_pred)  # On donne la vérité puis les prédictions

print(f"Log Model Accuracy: {log_accuracy:.2%}")

Log Model Accuracy: 92.05%


Up to now, everything that we've done are things we've done in previous labs - but now we'll get to try out some algorithms useful for reducing dimensionality! Let's use principal component analysis. Here, we'll reduce the space using enough axes to explain over 95% of the variability in the data...

## Q6) Import scikit's implementation of `PCA` and fit it to the training dataset so that 95% of the variability is explained.

*Hint 1: [Here is the documentation](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html) for scikit's `PCA` class.*

*Hint 2: [Here is the documentation](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html#sklearn.decomposition.PCA.fit_transform) for scikit's `.fit_transform()` method.*

In [48]:
# Ici on va réduire la dimensionnalité des données MNIST.
# D'abord, on importe la classe PCA de scikit-learn (module sklearn.decomposition)
# ------------------------------------------------------------------------------------------------------
from sklearn.decomposition import PCA # Importing PCA

In [49]:
# On crée l'objet PCA avec un hyperparamètre pour ne garder qu'un certain nombre de composantes
# Dans la doc : "Number of components to keep" -> n_components
# - si n_components est un ENTIER (ex: 50) : on garde exactement 50 composantes
# - si n_components est un FLOTTANT entre 0 et 1 (ex: 0.95) : on garde juste assez de composantes
#   pour expliquer au moins 95 % de la variance totale
# ---------------------------------------------------------------------------------------------------------------------------
pca = PCA(n_components=0.95) # Nombre de composantes choisi pour expliquer 95 % de la variabilité

In [50]:
# fit_transform = fit + transform :
#   fit       -> la PCA APPREND les axes principaux (directions de variance maximale) à partir des données d'entraînement
#   transform -> elle PROJETTE les données sur ces axes (on passe de 784 variables à ~150)
# ---------------------------------------------------------------------------------------------------------------------------
X_train_reduced = pca.fit_transform(X_train) # Fit-transform sur le jeu d'entraînement

# Combien de composantes ont été gardées pour atteindre 95 % de variance ?
print(f"Dimensions avant : {X_train.shape[1]}  ->  après : {X_train_reduced.shape[1]}")
print(f"Variance expliquée : {pca.explained_variance_ratio_.sum():.2%}")

Dimensions avant : 784  ->  après : 154
Variance expliquée : 95.04%


💡 **PCA en bref** : la PCA cherche de nouveaux axes (les *composantes principales*), combinaisons linéaires des pixels, classés par quantité de variance qu'ils expliquent. En gardant uniquement les premiers axes, on conserve l'essentiel de l'information avec beaucoup moins de variables. `n_components=0.95` demande à scikit-learn de choisir automatiquement combien d'axes il faut pour atteindre 95 % de variance expliquée.

In [51]:
# On applique la MÊME transformation au jeu de test, avec transform() uniquement (surtout pas fit !)
# Pourquoi ? Les axes doivent être appris sur l'entraînement seul. Refitter sur le test serait une
# "fuite de données" (data leakage) et les deux jeux ne seraient plus dans le même espace.
# ---------------------------------------------------------------------------------------------------------------------------
X_test_reduced = pca.transform(X_test)

## Q7) Repeat Q3 & Q4 using the *reduced* `X_train` dataset instead of `X_train`.

In [ ]:
# On relève l'heure, on ré-entraîne le RF avec X_train_reduced, on relève l'heure
# (rnd_clf.fit() ré-entraîne le modèle depuis zéro : l'ancien entraînement est écrasé)
# ------------------------------------------------------------------------------------------------------
t0 = time.time() # Heure avant l'entraînement
rnd_clf.fit(X_train_reduced, y_train) # Entraînement avec les données réduites
t1 = time.time()  # Heure après l'entraînement

# Combien de secondes pour entraîner le modèle ?
# ------------------------------------------------------------------------------------------------------
train_t_rf = t1-t0
print(f"Training took {train_t_rf:.2f}s")

# At every node, each tree searches for the best threshold on a feature,
# that best separates the digits. To do so it tries many candidate
# cut values and keeps the one giving the purest groups.
# - Raw pixels: integers 0-255, mostly 0 (black background), so there are very few distinct
#   values to try per feature -> cheap splits.
# - PCA components: dense, continuous values, almost every image has a different one, so there
#   are tens of thousands of candidate cut values per feature -> much more computation.
# Each component also mixes many pixels, so trees may need more splits to separate the digits.
# Conclusion: fewer features does not automatically mean faster training for tree-based models.

Training took 91.58s


In [ ]:
# Prédictions sur le jeu de test RÉDUIT (X_test_reduced) : le modèle a été entraîné sur des données réduites,
# il faut donc lui donner des données réduites aussi (sinon erreur de dimensions)
# ------------------------------------------------------------------------------------------------------
y_pred = rnd_clf.predict(X_test_reduced)

# Accuracy, 94.7 % 
# ------------------------------------------------------------------------------------------------------
red_rf_accuracy = accuracy_score(y_test, y_pred)  # Vérité et prédictions
print(f"RF Model Accuracy on reduced dataset: {red_rf_accuracy:.2%}")
# Accuracy is slightly lower after PCA because 5% of the variance (some useful details for
# separating similar digits) is discarded, and PCA keeps high-variance directions, not
# necessarily the most discriminative ones. 

RF Model Accuracy on reduced dataset: 94.70%


## Q8) Repeat Q5 using the *reduced* X_train dataset instead of X_train.

In [ ]:
# On relève l'heure, on ré-entraîne la régression logistique avec X_train_reduced, on relève l'heure
# ------------------------------------------------------------------------------------------------------
t0 = time.time() # Heure avant l'entraînement
log_clf.fit(X_train_reduced, y_train) # Entraînement avec les données réduites
t1 = time.time() # Heure après l'entraînement

# Combien de secondes pour entraîner le modèle ? 2.7 seconde 
# ------------------------------------------------------------------------------------------------------
train_t_log = t1-t0
print(f"Training took {train_t_log:.2f}s")
# Logistic regression is faster on the reduced data: each iteration is cheaper (about 5x fewer
# features and coefficients to handle) because regression is data * Nbr of number, and PCA components are uncorrelated, which makes the
# optimization easier for lbfgs. Its cost depends on the number of features, unlike trees. Les pixel cote a cote sont forcément corrélé. 

Training took 2.75s


/Users/estelle/miniforge3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [ ]:
# Prédictions sur le jeu de test réduit
# ------------------------------------------------------------------------------------------------------
y_pred = log_clf.predict(X_test_reduced)   # Prédictions à partir du jeu de test réduit

# Accuracy 91.38 % un peu moins meilleur que sans PCA
# ------------------------------------------------------------------------------------------------------
log_accuracy = accuracy_score(y_test, y_pred)  # Vérité et prédictions
print(f"Log Model Accuracy on reduced training data: {log_accuracy:.2%}")

Log Model Accuracy on reduced training data: 91.38%


You can now compare how well the random forest classifier and logistic regression classifier performed on both the full dataset and the reduced dataset. What were you able to observe? 

**Mes observations** (accuracies = celles indiquées dans le notebook ; pour les temps, compare avec tes propres valeurs affichées, elles dépendent de la machine) :

| Modèle | Données | Accuracy |
|---|---|---|
| Random Forest | complètes (784 variables) | ~96.7 % |
| Random Forest | réduites par PCA (~150 variables) | ~94.7 % |
| Régression logistique | complètes (784 variables) | ~92.1 % |
| Régression logistique | réduites par PCA (~150 variables) | ~91.4 % |

- **Précision** : la PCA fait perdre un peu de performance (environ -2 points pour le RF, moins d'1 point pour la régression logistique), ce qui est normal puisqu'on a jeté 5 % de la variance. Le Random Forest reste meilleur que la régression logistique dans les deux cas, car il capture des relations non linéaires entre pixels.
- **Réduction des données** : on passe d'environ 784 à ~150 variables (≈ 80 % en moins) tout en gardant 95 % de la variance : beaucoup de pixels sont redondants (bords de l'image toujours noirs, pixels voisins corrélés).
- **Temps d'entraînement** : la régression logistique est en général nettement plus rapide avec les données réduites (moins de variables à optimiser, et convergence plus facile). Pour le Random Forest, la réduction **n'accélère pas forcément** l'entraînement, et il peut même être plus lent : les pixels bruts sont des entiers avec beaucoup de zéros (faciles à découper pour un arbre), alors que les composantes PCA sont des variables continues denses, plus coûteuses à trier. **La réduction de dimension n'est donc pas un gain de temps garanti : cela dépend du modèle.**
- **Conclusion** : la PCA est utile pour compresser les données, accélérer certains algorithmes (surtout ceux sensibles au nombre de variables comme la régression logistique ou les SVM), visualiser et limiter le bruit, au prix d'une petite perte de précision et d'une perte d'interprétabilité (les composantes ne sont plus des pixels).
